# 03 · Recommendations engine: demonstração

A engine recebe **uma campanha** e devolve recomendações acionáveis, quantificadas e priorizadas. Para cada ação possível:

1. **Só ações que a campanha ainda não cumpre** (hook, rosto, CTA, legenda, densidade de texto, formato, áudio, duração). Horizontal nunca é sugerido no TikTok.
2. **Ganho no klike_score:** muda só aquele atributo, repreve com o **modelo A** e compara (contrafactual). Como o modelo é linear, o ganho é o coeficiente da notebook 02 já ajustado à plataforma.
3. **Evidência na métrica de negócio:** lift mediano de CTR (objetivos traffic, conversions, app_install) ou de engajamento (awareness, engagement) **na mesma plataforma**, com IC bootstrap de 90% e o n de cada grupo.
4. **Prioridade = ganho ÷ esforço** (baixo = 1, médio = 2, alto = 3). Duração é hipótese (D10) e vai sempre para o fim, como teste A/B sugerido.
5. **Plano combinado:** todas as ações firmes aplicadas juntas, com o score previsto antes e depois.

Código em `src/klike/recommender.py`; o mesmo resultado sai pela CLI (`python cli.py --campaign-id KLK-0127`).

In [1]:
from klike.data import load_clean
from klike.recommender import format_report, recommend

df = load_clean()
def show(campaign_id, top_k=5):
    print(format_report(recommend(df[df["campaign_id"] == campaign_id].iloc[0], top_k=top_k)))

## 1. TikTok · SaaS · conversões

Criativo sem hook, sem CTA e com muito texto na tela. A redução do texto vem em primeiro, à frente do hook, porque é barata: o hook vale mais pontos (+16,9 contra +12,9), mas exige reeditar a abertura.

In [2]:
show("KLK-0127")

Campanha KLK-0127 · TikTok · SaaS · objetivo conversions
Klike score previsto: 53,6 (real: 41,3)

1. Reduzir o texto na tela (densidade média): +12,9 pts no klike_score previsto (53,6 → 66,5). No TikTok, a diferença no CTR não é clara (+37%; IC 90%: −31% a +88%; n=94 vs 37). Como: Manter só a mensagem principal e o CTA em texto; o resto vai na fala ou na legenda. Esforço: baixo.
2. Adicionar um hook nos 3 primeiros segundos: +16,9 pts no klike_score previsto (53,6 → 70,5). No TikTok, campanhas assim têm CTR mediano +79% (IC 90%: +34% a +134%; n=83 vs 93). Como: Abrir com a promessa, o problema ou um movimento forte, antes de qualquer logo. Esforço: médio.
3. Incluir um CTA explícito: +8,0 pts no klike_score previsto (53,6 → 61,6). No TikTok, a diferença no CTR não é clara (+15%; IC 90%: −18% a +69%; n=112 vs 64). Como: Dizer e escrever o próximo passo ("Baixe agora", "Peça uma demo"). Esforço: baixo.
4. Testar uma versão de 16–30s (hipótese a testar): −0,1 pts no score previsto. No Tik

## 2. Meta · Lead Gen · engajamento

Aqui a evidência na métrica de negócio é fraca para todas as ações (os ICs cruzam o zero): o ganho é no score, e a engine diz isso em vez de prometer engajamento. A duração entra como hipótese, porque o vídeo tem 29s e a faixa com melhor CTR no Meta foi 11–15s.

In [3]:
show("KLK-0348")

Campanha KLK-0348 · Meta · Lead Gen · objetivo engagement
Klike score previsto: 65,1 (real: 38,4)

1. Aumentar um pouco o texto na tela (densidade média): +7,4 pts no klike_score previsto (65,1 → 72,5). No Meta, a diferença no Engajamento não é clara (+23%; IC 90%: −9% a +55%; n=96 vs 97). Como: Reforçar a mensagem principal e o CTA com texto curto na tela. Esforço: baixo.
2. Adicionar legendas: +7,3 pts no klike_score previsto (65,1 → 72,4). No Meta, a diferença no Engajamento não é clara (−2%; IC 90%: −27% a +17%; n=119 vs 86). Como: Legendar a fala: boa parte do público assiste sem som. Esforço: baixo.
3. Exportar em formato quadrado: +4,9 pts no klike_score previsto (65,1 → 70,0). No Meta, a diferença no Engajamento não é clara (+17%; IC 90%: −11% a +62%; n=59 vs 63). Como: Reenquadrar o vídeo para o formato nativo do feed desta plataforma. Esforço: médio.
4. Testar uma versão de 11–15s (hipótese a testar): +0,3 pts no score previsto. No Meta, campanhas assim têm CTR mediano +32% (

## 3. LinkedIn · SaaS · conversões

No LinkedIn o hook vale mais que nas outras plataformas (+19,4 pts) e tem a evidência mais forte de CTR. O formato recomendado é **horizontal**, o contrário do TikTok, e a engine avisa que não há campanhas verticais suficientes no LinkedIn para medir o efeito no CTR (n=11).

In [4]:
show("KLK-0262")

Campanha KLK-0262 · LinkedIn · SaaS · objetivo conversions
Klike score previsto: 47,6 (real: 46,7)

1. Adicionar um hook nos 3 primeiros segundos: +19,4 pts no klike_score previsto (47,6 → 66,9). No LinkedIn, campanhas assim têm CTR mediano +119% (IC 90%: +29% a +239%; n=45 vs 38). Como: Abrir com a promessa, o problema ou um movimento forte, antes de qualquer logo. Esforço: médio.
2. Aumentar um pouco o texto na tela (densidade média): +8,7 pts no klike_score previsto (47,6 → 56,2). No LinkedIn, a diferença no CTR não é clara (+3%; IC 90%: −29% a +90%; n=31 vs 30). Como: Reforçar a mensagem principal e o CTA com texto curto na tela. Esforço: baixo.
3. Mostrar um rosto humano: +11,6 pts no klike_score previsto (47,6 → 59,1). No LinkedIn, a diferença no CTR não é clara (+62%; IC 90%: −2% a +132%; n=48 vs 35). Como: Incluir uma pessoa falando para a câmera, idealmente já na abertura. Esforço: alto.
4. Exportar em formato horizontal: +4,7 pts no klike_score previsto (47,6 → 52,3). Sem cam

## 4. Criativo novo, antes do lançamento

O uso principal do Klike: o vídeo ainda não rodou, então não há CTR nem ROAS. Basta descrever o criativo e o contexto.

In [5]:
new_creative = {
    "platform": "TikTok", "category": "E-commerce", "objective": "traffic", "target_audience_age": "18-24",
    "is_retargeting": False, "video_duration_s": 40, "format": "quadrado", "has_subtitle": None,
    "has_cta": False, "has_hook": False, "has_face": False, "text_density": "high", "music_voice_ratio": 0.8,
}
print(format_report(recommend(new_creative)))

Campanha (nova) · TikTok · E-commerce · objetivo traffic
Klike score previsto: 23,5

1. Reduzir o texto na tela (densidade média): +12,9 pts no klike_score previsto (23,5 → 36,3). No TikTok, a diferença no CTR não é clara (+37%; IC 90%: −31% a +88%; n=94 vs 37). Como: Manter só a mensagem principal e o CTA em texto; o resto vai na fala ou na legenda. Esforço: baixo.
2. Adicionar um hook nos 3 primeiros segundos: +16,9 pts no klike_score previsto (23,5 → 40,4). No TikTok, campanhas assim têm CTR mediano +79% (IC 90%: +34% a +134%; n=83 vs 93). Como: Abrir com a promessa, o problema ou um movimento forte, antes de qualquer logo. Esforço: médio.
3. Incluir um CTA explícito: +8,0 pts no klike_score previsto (23,5 → 31,5). No TikTok, a diferença no CTR não é clara (+15%; IC 90%: −18% a +69%; n=112 vs 64). Como: Dizer e escrever o próximo passo ("Baixe agora", "Peça uma demo"). Esforço: baixo.
4. Adicionar legendas: +7,8 pts no klike_score previsto (23,5 → 31,3). No TikTok, a diferença no CT

## 5. Criativo que já cumpre as boas práticas

A engine não inventa recomendações: com hook, rosto, CTA, legenda e densidade média, sobra só o teste de duração.

In [6]:
show("KLK-0003")

Campanha KLK-0003 · TikTok · SaaS · objetivo traffic
Klike score previsto: 90,0 (real: 99,1)

O criativo já cumpre a maior parte das boas práticas mapeadas; restam poucas melhorias.

1. Testar uma versão de 16–30s (hipótese a testar): −0,3 pts no score previsto. No TikTok, campanhas assim têm CTR mediano +75% (IC 90%: +12% a +118%; n=46 vs 115). Como: Hoje o vídeo tem 8s. Faixa com melhor CTR no TikTok na base; vale um teste A/B. Esforço: médio.


## 6. Saída estruturada (JSON)

Para integrar com o produto, a mesma recomendação sai como dicionário/JSON (`python cli.py --campaign-id KLK-0127 --json`).

In [7]:
import json
result = recommend(df[df["campaign_id"] == "KLK-0127"].iloc[0], top_k=1)
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

{
  "campaign_id": "KLK-0127",
  "platform": "TikTok",
  "category": "SaaS",
  "objective": "conversions",
  "score_actual": 41.3,
  "score_predicted": 53.6,
  "n_available": 4,
  "recommendations": [
    {
      "key": "text",
      "title": "Reduzir o texto na tela (densidade média)",
      "how": "Manter só a mensagem principal e o CTA em texto; o resto vai na fala ou na legenda.",
      "change": {
        "attribute": "text_density",
        "from": "high",
        "to": "medium"
      },
      "score_gain": 12.9,
      "score_after": 66.5,
      "effort": "baixo",
      "priority": 12.87,
      "evidence": {
        "metric": "ctr",
        "context": "no TikTok",
        "n_treat": 94,
        "n_ctrl": 37,
        "lift": 0.3685,
        "ci_low": -0.3118,
        "ci_high": 0.8819,
        "median_treat": 0.0583,
        "median_ctrl": 0.0426
      },
      "hypothesis": false,
      "text": "Reduzir o texto na tela (densidade média): +12,9 pts no klike_score previsto (53,6 → 

## Limitações

- **Associação, não causa.** Os ganhos vêm de 500 campanhas observadas. Um criativo com hook pode ter outras qualidades que a base não mede. A validação certa é um teste A/B por recomendação.
- **O score previsto erra ~6,6 pts em média** (MAE do modelo A). Ganhos de 1–2 pts estão dentro do ruído.
- **O plano combinado soma os efeitos** porque o modelo é linear; na prática, as ações podem ter retorno decrescente (o 5º acerto vale menos que o 1º), o que a base não permite estimar.
- **Categoria não entra na evidência empírica**: com ~30 campanhas por plataforma × categoria, os ICs ficam largos demais. Ela entra no score previsto, pelo modelo.